# Notebook 2 — FLAN-T5-base: Financial Domain Prompt

**Model:** `google/flan-t5-base` (~250M parameters)  
**Task:** Text simplification using a detailed, financial domain-specific prompt  
**Input:** `data/final/dataset_model_v2.csv` (must already contain `output_generic` from Notebook 1)  
**Output column:** `output_financial`  
**Saved to:** `data/final/dataset_model_v2.csv`

---
This notebook tests whether a **more specific prompt** improves results on the same base model.  
The prompt explicitly mentions the financial domain and instructs the model to preserve numbers, dates, and legal conditions.

## 0. Environment Info

In [ ]:
# Environment info — run this first to log versions for reproducibility
import platform, importlib
for pkg in ["torch", "transformers", "sentencepiece"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<15}: {mod.__version__}")
    except Exception:
        print(f"{pkg:<15}: not installed")
print(f"{'python':<15}: {platform.python_version()}")
print(f"{'platform':<15}: {platform.platform()}")

## 1. Install Dependencies

In [ ]:
import subprocess, sys

packages = ['transformers==4.40.0', 'torch==2.2.0', 'sentencepiece==0.1.99']
for pkg in packages:
    base = pkg.split('==')[0].replace('-', '_')
    try:
        __import__(base)
        print(f'  {pkg} already installed')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  {pkg} installed')

## 2. Imports

In [ ]:
import pandas as pd
import torch
import re
from transformers import T5ForConditionalGeneration, T5Tokenizer

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device    : {DEVICE}")

## 2b. Set Random Seed

In [ ]:
import torch, random, numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed set to {SEED}")

## 3. Load Dataset

In [ ]:
DATA_PATH = "../data/final/dataset_model_v2.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

assert "output_generic" in df.columns, "Run Notebook 1 first to generate output_generic!"
df[["candidate_id", "text", "output_generic"]].head(3)

## 4. Load FLAN-T5-base Model

In [ ]:
model_name = "google/flan-t5-base"

print(f"Loading tokenizer: {model_name}")
tokenizer = T5Tokenizer.from_pretrained(model_name)

print(f"Loading model: {model_name}")
model = T5ForConditionalGeneration.from_pretrained(model_name)
model = model.to(DEVICE)
model.eval()
print("Model loaded and ready.")

## 5. Define Prompt

> **Financial domain prompt** — explicitly instructs the model to act as a financial simplification expert and to preserve key information exactly.

In [ ]:
PROMPT_PREFIX = (
    "You are a financial document simplification expert. "
    "Rewrite the following financial or regulatory text in plain English for a non-expert consumer. "
    "Keep all numbers, dates, and legal conditions exactly as written. "
    "Do not add any information not in the original. "
    "Preserve the original meaning exactly:\n\n"
)

example_prompt = PROMPT_PREFIX + str(df["text"].iloc[0])
print("=== Example Prompt ===")
print(example_prompt)
print(f"\nToken count: {len(tokenizer(example_prompt)['input_ids'])}")
print(f"\nPrompt prefix alone: {len(tokenizer(PROMPT_PREFIX)['input_ids'])} tokens")

## 6. Run Inference

In [ ]:
outputs = []
total = len(df)

for i, row in df.iterrows():
    prompt = PROMPT_PREFIX + str(row["text"])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    outputs.append(decoded)

    row_num = i + 1
    if row_num % 10 == 0:
        print(f"Processed {row_num}/{total} rows...")

print(f"Processed {total}/{total} rows. Done.")

## 7. Save Results

In [ ]:
df["output_financial"] = outputs
df.to_csv(DATA_PATH, index=False)
print(f"Saved to {DATA_PATH}")
print(f"Columns: {df.columns.tolist()}")

## 8. Summary Statistics

In [ ]:
def approx_fkgl(text):
    words = str(text).split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', str(text)) if s.strip()]
    num_sentences = max(len(sentences), 1)
    num_words = max(len(words), 1)
    num_syllables = sum(max(1, len(w) // 3) for w in words)
    return round(0.39 * (num_words / num_sentences) + 11.8 * (num_syllables / num_words) - 15.59, 2)

df["wc_financial"]   = df["output_financial"].apply(lambda x: len(str(x).split()))
df["fkgl_financial"] = df["output_financial"].apply(approx_fkgl)
identical = (df["output_financial"].str.strip() == df["text"].str.strip()).sum()

print("=" * 50)
print("SUMMARY — FLAN-T5-base Financial Domain Prompt")
print("=" * 50)
print(f"Total rows processed        : {len(df)}")
print(f"Avg word count (output)     : {df['wc_financial'].mean():.2f}")
print(f"Avg FKGL (output)           : {df['fkgl_financial'].mean():.2f}")
print(f"Rows identical to input     : {identical}/{len(df)}")

## 9. Compare with Generic Prompt (Notebook 1)

## 10. Key Finding: Why More Identical Rows?

The financial prompt is **longer** (~80 tokens), leaving less room for the actual text within the 512-token limit.

In [ ]:
GENERIC_PREFIX = "Simplify the following text so that a non-expert can understand it:\n\n"

generic_prefix_tokens   = len(tokenizer(GENERIC_PREFIX)["input_ids"])
financial_prefix_tokens = len(tokenizer(PROMPT_PREFIX)["input_ids"])

print(f"Generic prompt prefix   : {generic_prefix_tokens} tokens")
print(f"Financial prompt prefix : {financial_prefix_tokens} tokens")
print(f"Extra tokens used by financial prompt: {financial_prefix_tokens - generic_prefix_tokens}")
print(f"Remaining token budget for text (financial): {512 - financial_prefix_tokens}")

df["token_count_financial"] = df["text"].apply(
    lambda t: len(tokenizer(PROMPT_PREFIX + str(t))["input_ids"])
)
truncated = (df["token_count_financial"] > 512).sum()
print(f"\nRows truncated (>512 tokens) with financial prompt: {truncated}/{len(df)}")

---
## Notes

- The financial prompt **increases** the number of identical-to-input rows compared to the generic prompt (36 vs 15). This is **counter-intuitive** but explained by token budget: the longer prompt leaves less space for the actual text.
- This demonstrates that **prompt engineering has diminishing returns on small models**. The model does not have sufficient capacity to follow complex instructions reliably.
- Neither generic nor financial prompt meaningfully reduces FKGL — both produce outputs at roughly the same reading level as the original.
- See `notebook_03` for the larger FLAN-T5-XL model with the same financial prompt.